# 03-4. ベイズの定理 — 動かして確かめる

📖 解説: [`../04_bayes.md`](../04_bayes.md)

## このノートで触るもの
1. ベイズの定理の基本: 病気の検査
2. ベイズ更新: コインの偏りを推定
3. 事前 → 事後 の段階的更新を可視化
4. ナイーブベイズ分類器の小実装

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (04_bayes.md)](../04_bayes.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from scipy import stats
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline
rng = np.random.default_rng(42)

## 1. ベイズの定理 — 病気検査

$$
P(D \mid +) = \frac{P(+ \mid D) \cdot P(D)}{P(+)}
$$

対応するコード: `(p_pos_given_d * p_d) / p_pos`

In [ ]:
def bayes_disease_test(prior_disease: float, sensitivity: float, specificity: float) -> None:
    """病気検査のベイズ計算."""
    p_d = prior_disease
    p_pos_d = sensitivity
    p_pos_notd = 1 - specificity
    
    # 全確率
    p_pos = p_pos_d * p_d + p_pos_notd * (1 - p_d)
    
    # ベイズ
    p_d_pos = (p_pos_d * p_d) / p_pos
    
    print(f'事前確率 P(病気)         = {p_d:.4f} ({p_d*100:.1f}%)')
    print(f'感度   P(+ | 病気)       = {p_pos_d:.4f}')
    print(f'特異度 P(- | 健康)       = {specificity:.4f}')
    print(f'-----------------------------------')
    print(f'陽性率 P(+)              = {p_pos:.4f}')
    print(f'★ 事後確率 P(病気 | +)  = {p_d_pos:.4f} ({p_d_pos*100:.2f}%)')
    print()
    if p_d_pos < 0.5:
        print(f'→ 陽性が出ても、病気である確率は {p_d_pos*100:.1f}% しかない!')
        print('  これが「基底率の誤謬」: 病気の事前確率が低いと、検査の精度が高くても誤陽性が支配的')

interact(bayes_disease_test,
         prior_disease=FloatSlider(min=0.001, max=0.5, step=0.001, value=0.01, description='事前確率'),
         sensitivity=FloatSlider(min=0.5, max=1.0, step=0.01, value=0.99, description='感度'),
         specificity=FloatSlider(min=0.5, max=1.0, step=0.01, value=0.99, description='特異度'));

## 2. ベイズ更新: コインの偏り推定

事前 $P(\theta) = U(0, 1)$

尤度 $P(D \mid \theta) = \theta^k (1-\theta)^{n-k}$

事後 $P(\theta \mid D) \propto \theta^k (1-\theta)^{n-k}$ ← Beta 分布

In [ ]:
true_p = 0.7   # 本当のコインの偏り (普段は分からない)
rng_local = np.random.default_rng(42)

theta_grid = np.linspace(0.001, 0.999, 200)

fig, ax = plt.subplots(figsize=(11, 6))

for n_flips in [0, 5, 20, 100, 500]:
    if n_flips == 0:
        # 事前: 一様
        posterior = np.ones_like(theta_grid)
    else:
        flips = rng_local.binomial(1, true_p, n_flips - (0 if n_flips == 5 else 5))
        # 累積データを使うため、再生成じゃなく continued sampling にすべきだが
        # ここではデモ用に毎回 n_flips 回投げる
        flips = rng_local.binomial(1, true_p, n_flips)
        k = flips.sum()
        likelihood = theta_grid**k * (1 - theta_grid)**(n_flips - k)
        posterior = likelihood
    
    posterior = posterior / posterior.sum() / (theta_grid[1] - theta_grid[0])  # 規格化
    ax.plot(theta_grid, posterior, label=f'n = {n_flips}')

ax.axvline(true_p, color='red', linestyle='--', label=f'真の値 $\\theta = {true_p}$')
ax.set_xlabel('$\\theta$ (表が出る確率)')
ax.set_ylabel('事後密度')
ax.set_title('ベイズ更新: データが増えると事後分布が真の値に集中')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 3. 段階的にデータを増やしながら MAP 推定

MAP (Maximum a Posteriori): 事後確率を最大化する $\theta$

In [ ]:
true_p = 0.7
rng_local = np.random.default_rng(42)

n_total = 500
all_flips = rng_local.binomial(1, true_p, n_total)

theta_grid = np.linspace(0.001, 0.999, 500)
ns = []
maps = []

for n in range(1, n_total + 1):
    k = all_flips[:n].sum()
    likelihood = theta_grid**k * (1 - theta_grid)**(n - k)
    map_estimate = theta_grid[likelihood.argmax()]
    ns.append(n)
    maps.append(map_estimate)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ns, maps, label='MAP 推定')
ax.axhline(true_p, color='red', linestyle='--', label=f'真の値 {true_p}')
ax.set_xlabel('試行回数 n')
ax.set_ylabel('推定 $\\theta$')
ax.set_title('データが増えると MAP 推定は真の値に収束')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. ナイーブベイズ分類器 (簡易スパムフィルタ)

$$
P(\text{spam} \mid w_1, \dots, w_n) \propto P(\text{spam}) \cdot \prod_i P(w_i \mid \text{spam})
$$

In [ ]:
# 訓練データ (簡易版)
spam_messages = [
    ['win', 'free', 'money'],
    ['free', 'iphone', 'click'],
    ['win', 'lottery', 'claim'],
    ['discount', 'free', 'now'],
]
ham_messages = [
    ['meeting', 'tomorrow', 'office'],
    ['lunch', 'with', 'team'],
    ['report', 'attached', 'review'],
    ['project', 'update', 'meeting'],
]

from collections import Counter

# 単語ごとの確率を計算
spam_words = Counter([w for msg in spam_messages for w in msg])
ham_words = Counter([w for msg in ham_messages for w in msg])

vocab = set(spam_words.keys()) | set(ham_words.keys())
n_spam = sum(spam_words.values())
n_ham = sum(ham_words.values())

# Laplace smoothing 付き
def p_word_given_class(word, word_count, total, vocab_size):
    return (word_count.get(word, 0) + 1) / (total + vocab_size)

def classify(message: list[str]) -> tuple[str, float]:
    """ナイーブベイズで分類."""
    p_spam_prior = len(spam_messages) / (len(spam_messages) + len(ham_messages))
    p_ham_prior = 1 - p_spam_prior
    
    # 対数確率で計算 (アンダーフロー回避)
    log_p_spam = np.log(p_spam_prior)
    log_p_ham = np.log(p_ham_prior)
    
    for word in message:
        log_p_spam += np.log(p_word_given_class(word, spam_words, n_spam, len(vocab)))
        log_p_ham += np.log(p_word_given_class(word, ham_words, n_ham, len(vocab)))
    
    # ソフトマックスで確率に
    max_log = max(log_p_spam, log_p_ham)
    p_spam = np.exp(log_p_spam - max_log)
    p_ham = np.exp(log_p_ham - max_log)
    p_spam_norm = p_spam / (p_spam + p_ham)
    
    return ('spam' if p_spam_norm > 0.5 else 'ham', p_spam_norm)

# テスト
test_messages = [
    ['win', 'free', 'iphone'],
    ['meeting', 'with', 'team'],
    ['free', 'report'],
    ['lunch', 'tomorrow'],
]

for msg in test_messages:
    label, p = classify(msg)
    print(f'{str(msg):<40} → {label}  (P(spam) = {p:.4f})')

## まとめ
- ベイズの定理は条件付き確率を「ひっくり返す」道具
- 病気検査のシミュレータで「基底率の誤謬」を体感
- ベイズ更新でデータが増えると事後が真値に集中
- ナイーブベイズは ML の最古典分類器

## 確率・統計章 完了 🎉

→ 次の章: [`../../05_optimization/README.md`](../../05_optimization/README.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次の章 → |
|---|---|---|---|
| [`03_expectation_variance.ipynb`](03_expectation_variance.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`../../05_optimization/README.md`](../../05_optimization/README.md) |